# **Langchain basics** ⛓



---
A basic introduction to using langchain with OpenAI api and using basic Rag

Langchain tutorial:

https://www.youtube.com/watch?v=yF9kGESAi3M


## **Setting up the environment**

In [3]:
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_community.callbacks import get_openai_callback

import os

In [4]:
load_dotenv()  # Automatically searches for .env in current and parent directories

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
goole_api_key = os.getenv("GOOGLE_API_KEY")


In [5]:
for key, value in [("OPENAI_API_KEY", openai_api_key), ("ANTHROPIC_API_KEY", anthropic_api_key), ("GOOGLE_API_KEY", goole_api_key)]:
    if value is not None:
        print(f"Environment variable {key} is set.")
    else:
        print(f"Environment variable {key} is NOT set.")

Environment variable OPENAI_API_KEY is set.
Environment variable ANTHROPIC_API_KEY is NOT set.
Environment variable GOOGLE_API_KEY is NOT set.




---


## **Simple chatbot inference over langchain**

In [13]:
# Create a ChatOpenAI model
llm = ChatOpenAI(model="gpt-5.2-chat-latest")

In [ ]:
# Initialize chat history with system message
chat_history = [SystemMessage(content="You are a helpful AI assistant.")]

def chat(user_message):
    """Send a message and get a response"""
    # Add user message
    chat_history.append(HumanMessage(content=user_message))
    
    # Get AI response
    result = llm.invoke(chat_history)
    response = result.content
    
    # Add AI response to history
    chat_history.append(AIMessage(content=response))
    
    print(f"You: {user_message}")
    print(f"\nAI: {response}\n")
    
    return response

# Example usage - modify the message and run
chat("Hello! What can you help me with?")

You:
What is Langchain?

AI: 
**LangChain** is an open‑source framework designed to help developers build applications powered by **large language models (LLMs)** like GPT‑4, Claude, or open‑source models.

At its core, LangChain makes it easier to **connect LLMs with external data, tools, and workflows**, enabling more complex and useful AI applications than simple prompt‑response interactions.

---

### What LangChain Is Used For
LangChain is commonly used to build:

- **Chatbots and AI assistants**
- **Question‑answering systems over documents**
- **Agents that can use tools (APIs, calculators, search engines)**
- **Automated workflows and decision‑making systems**
- **Retrieval‑augmented generation (RAG) applications**

---

### Key Concepts in LangChain

1. **LLMs & Chat Models**
   - Unified interfaces for calling models like OpenAI, Anthropic, Hugging Face, etc.

2. **Prompts**
   - Templates that structure inputs to LLMs dynamically.

3. **Chains**
   - Sequences of steps (e.g.

In [20]:
from pprint import pprint

print("---- Message History ----")
pprint(chat_history[-1:])

---- Message History ----
[AIMessage(content='**LangChain** is an open‑source framework designed to help developers build applications powered by **large language models (LLMs)** like GPT‑4, Claude, or open‑source models.\n\nAt its core, LangChain makes it easier to **connect LLMs with external data, tools, and workflows**, enabling more complex and useful AI applications than simple prompt‑response interactions.\n\n---\n\n### What LangChain Is Used For\nLangChain is commonly used to build:\n\n- **Chatbots and AI assistants**\n- **Question‑answering systems over documents**\n- **Agents that can use tools (APIs, calculators, search engines)**\n- **Automated workflows and decision‑making systems**\n- **Retrieval‑augmented generation (RAG) applications**\n\n---\n\n### Key Concepts in LangChain\n\n1. **LLMs & Chat Models**\n   - Unified interfaces for calling models like OpenAI, Anthropic, Hugging Face, etc.\n\n2. **Prompts**\n   - Templates that structure inputs to LLMs dynamically.\n\n3.



---



## **RAG - Retrieval Augmented Generation**

A short introduction to RAG:

https://www.youtube.com/watch?v=HREbdmOSQ18


Langchain tutorial:

https://python.langchain.com/docs/tutorials/rag/


In [21]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

Get a paper from Gilles from this side: https://profiles.ucl.ac.uk/55488-gilles-retsin/publications

Download a paper for a testrun

In [22]:
if not os.path.exists("./documents"):
    os.makedirs("./documents")

In [23]:
#get the pdf file from this link https://discovery.ucl.ac.uk/id/eprint/10117335/1/Toward_Discrete_Architecture_Automation.pdf
import requests

def download_pdf(url, filename):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an exception for bad status codes

        with open(filename, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print(f"PDF downloaded successfully as {filename}")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading PDF: {e}")

pdf_url = "https://discovery.ucl.ac.uk/id/eprint/10117335/1/Toward_Discrete_Architecture_Automation.pdf"
pdf_filename = "./documents/Toward_Discrete_Architecture_Automation.pdf"  # Choose your desired filename

download_pdf(pdf_url, pdf_filename)


PDF downloaded successfully as ./documents/Toward_Discrete_Architecture_Automation.pdf


In [24]:
# Define the directory containing the text file and the persistent directory
doc_name = "Toward_Discrete_Architecture_Automation.pdf"

file_path = os.path.join("./documents", doc_name)
persistent_directory = os.path.join("db", "chroma_db")



---


Split the text of the pdf in chunks and create a vector database

In [25]:
from langchain_community.document_loaders import PyPDFLoader


# Check if the Chroma vector store already exists
if not os.path.exists(persistent_directory):
    print("Persistent directory does not exist. Initializing vector store...")

    # Ensure the text file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"The file {file_path} does not exist. Please check the path."
        )

    # Read the text content from the file
    loader = PyPDFLoader(file_path)

    documents = []

    async for page in loader.alazy_load():
        documents.append(page)

    #documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(documents)

    # Display information about the split documents
    print("\n--- Document Chunks Information ---")
    print(f"Number of document chunks: {len(docs)}")
    print(f"Sample chunk:\n{docs[0].page_content}\n")

    # Create embeddings
    print("\n--- Creating embeddings ---")
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )  # Update to a valid embedding model if needed
    print("\n--- Finished creating embeddings ---")

    # Create the vector store and persist it automatically
    print("\n--- Creating vector store ---")
    db = Chroma.from_documents(
        docs, embeddings, persist_directory=persistent_directory)
    print("\n--- Finished creating vector store ---")

else:
    print("Vector store already exists. No need to initialize.")

Persistent directory does not exist. Initializing vector store...

--- Document Chunks Information ---
Number of document chunks: 41
Sample chunk:
TOPIC  (ACADIA team will fill in)
1
Toward Discrete Architecture: 
Automation takes Command
1 Gilles Retsin Architecture: 
Diamons House, Multifamily 
Home in Belgium (2015)
Gilles Retsin
UCL the Bartlett School of 
Architecture
1
ABSTRACT
This paper describes a framework for discrete computational design and fabrication in 
the context of automation. Whereas digital design and fabrication are technical notions, 
automation immediately has societal and political repercussions. Automation relates to 
industrialization and mechanisation—allowing to historically reconnect the digital while 
bypassing the  post-modern, deconstructivist, or parametric decades. Using a series of 
built prototypes making use of timber, this paper will describe how the combined tech-
nologies of automation and discreteness enable both technical efficiencies and new 



---


Perform a test query on the vector database

In [ ]:
# Function to query a vector store
def query_vector_store(store_name, query, embedding_function):
    if os.path.exists(persistent_directory):
        print(f"\n--- Querying the Vector Store {store_name} ---")

        db = Chroma(
            persist_directory=persistent_directory,
            embedding_function=embedding_function,
        )

        retriever = db.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={"k": 5, "score_threshold": 0.1},
        )

        relevant_docs = retriever.invoke(query)
        # Display the relevant results with metadata
        print(f"\n--- Relevant Documents for {store_name} ---")
        for i, doc in enumerate(relevant_docs, 1):
            print(f"Document {i}:\n{doc.page_content}\n")
            if doc.metadata:
                print(f"Source: {doc.metadata.get('source', 'Unknown')}\n")
    else:
        print(f"Vector store {store_name} does not exist.")

    return relevant_docs, retriever


# Define the user's question
query = "What is discrete architecture?"

# Query each vector store
relevant_docs, retriever = query_vector_store("chroma_db_openai", query, embeddings)


--- Querying the Vector Store chroma_db_openai ---


C:\Users\chris\AppData\Local\Temp\ipykernel_30308\208711191.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(



--- Relevant Documents for chroma_db_openai ---
Document 1:
project with a radical social, aesthetic, and spatial agenda. 
Discreteness plays a key technical role in automation, 
enabling both fast assembly and complexity (Gerschenfeld 
et al. 2015). Besides these technical concerns, the Discrete 
has far-reaching architectural ambitions, exploring the 
myriad implications and consequences of an architecture 
based on autonomous parts, rather than super-imposing 
whole.
Prefabrication and Programmable Matter
This work on discrete building blocks and robotics relates 
to Hod Lipson’s Programmable Matter research, MIT’s 
CIAL modular robotics, and the Digital Material Research 
at the Centre for Bits and Atoms, with projects such as 
Flexural Materials by Kenneth Cheung, or Bill-E by Benjamin 
Jennett. The Harvard Wyss Institute's work on the TERMES 
project  is another reference, where a distributed robot 
assembles serialised building blocks. While these examples 
are mainly situated 



---

## **Simple chatbot with RAG inference**

Loading the packages

In [28]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [31]:
# Answer question prompt
qa_system_prompt = (
    "You are an assistant for question-answering tasks. Use "
    "the following pieces of retrieved context to answer the "
    "question. If you don't know the answer, just say that you "
    "don't know. Use three sentences maximum and keep the answer "
    "concise."
    "\n\n"
    "{context}"
)

# Create QA prompt template
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Build the RAG chain using LCEL with proper input mapping
rag_chain = (
    {
        "context": lambda x: "\n\n".join([doc.page_content for doc in retriever.invoke(x["input"])]),
        "input": lambda x: x["input"],
        "chat_history": lambda x: x.get("chat_history", [])
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

Create a template and instances of the Q&A chain and the RAG chain

In [32]:
# Initialize RAG chat history
rag_chat_history = []

def rag_chat(user_message):
    """Send a message and get a RAG-based response"""
    # Add user message to history
    rag_chat_history.append(HumanMessage(content=user_message))
    
    # Invoke the RAG chain
    result = rag_chain.invoke({
        "input": user_message,
        "chat_history": rag_chat_history
    })
    
    response = result
    
    # Add AI response to history
    rag_chat_history.append(AIMessage(content=response))
    
    print(f"You: {user_message}")
    print(f"\nAI: {response}\n")
    
    return response

# Example usage - modify the query and run
rag_chat("What does the paper discuss about discrete architecture?")

You: What does the paper discuss about discrete architecture?

AI: The paper discusses discrete architecture as a framework combining automation and discrete building elements to enable efficient, flexible, and scalable construction. It explores how autonomous, modular parts—often assembled with robotic automation—can replace monolithic architectural systems while drawing from concepts like programmable matter and prefabrication. Beyond technical efficiency, it positions discreteness as having social, political, and architectural implications that challenge parametric and post-digital design trends.



'The paper discusses discrete architecture as a framework combining automation and discrete building elements to enable efficient, flexible, and scalable construction. It explores how autonomous, modular parts—often assembled with robotic automation—can replace monolithic architectural systems while drawing from concepts like programmable matter and prefabrication. Beyond technical efficiency, it positions discreteness as having social, political, and architectural implications that challenge parametric and post-digital design trends.'

In [38]:
def continual_chat():
    """Interactive chat with RAG system"""
    print("Start chatting with the AI! Type 'exit' to end the conversation.")
    chat_history = []  # Collect chat history here
    while True:
        query = input("\nYou: ")
        if query.lower() == "exit":
            break
        print("You:\n" + query+"\n")
        # Add user message to history
        chat_history.append(HumanMessage(content=query))
        
        # Process the user's query through the RAG chain
        result = rag_chain.invoke({"input": query, "chat_history": chat_history})
        
        # Display the AI's response (result is a string, not a dict)
        print(f"\nAI: {result}\n")
        
        # Add AI response to history
        chat_history.append(AIMessage(content=result))

Test questions:

*   Please explain the theoretical framework of discrete architecture
*   Please elaborate on the importance of automation in the discrete concept

In [39]:
continual_chat()

Start chatting with the AI! Type 'exit' to end the conversation.
You:
Please explain the theoretical framework of discrete architecture


AI: Discrete architecture is a theoretical framework that centers on buildings composed of autonomous, standardized parts rather than continuous, monolithic forms. It links digital design with automation and robotics, enabling fast assembly, adaptability, and complexity while reconnecting architecture to industrialization and societal implications of automation. Politically and critically, it positions itself against parametric and post-digital trends by emphasizing discreteness as both a technical and architectural agenda.

You:
Please elaborate on the importance of automation in the discrete concept


AI: Automation is essential to the discrete concept because it enables the efficient assembly of standardized building blocks, making architectural complexity achievable through simple, repeatable operations. It shifts digital fabrication toward a bro